# Week 13 live coding: Education policy data

Mục tiêu: biến policy sources thành bảng metadata + coding scheme có thể kiểm tra, đếm, vẽ timeline và viết Data/Methods paragraph thận trọng.

Core tuần này: chạy notebook, đọc policy-area summary + source-type summary, xuất timeline/figure, và viết caption + source note + Data/Methods paragraph.

In [1]:
import hashlib
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve
try:
    import pandas as pd
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas==2.3.3", "matplotlib==3.9.4"])
    import pandas as pd
    import matplotlib.pyplot as plt
plt.rcParams["svg.hashsalt"] = "week13-education-policy-data"
print("pandas:", pd.__version__)
WEEK_PATH = Path("weeks/week-13-education-policy-data")
WEEK_DIR = WEEK_PATH if WEEK_PATH.exists() else Path(".")
DATA_PATH = WEEK_DIR / "data" / "raw" / "week13_policy_coding.csv"
TABLE_DIR = WEEK_DIR / "outputs" / "tables"
FIG_DIR = WEEK_DIR / "outputs" / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA = "cedbc768526aadf2ebfe41fdf08585ebe10963e5ba17adb1a78f93ca14d8aaa5"
REMOTE_DATA = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-13-education-policy-data/data/raw/week13_policy_coding.csv"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(REMOTE_DATA, DATA_PATH)
actual_sha = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Data file:", DATA_PATH)
print("SHA-256:", actual_sha)
if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "Data SHA differs from the course snapshot. "
        "Stop and check whether you intentionally changed week13_policy_coding.csv."
    )

pandas: 2.3.3
Data file: weeks/week-13-education-policy-data/data/raw/week13_policy_coding.csv
SHA-256: cedbc768526aadf2ebfe41fdf08585ebe10963e5ba17adb1a78f93ca14d8aaa5


## 1. Read policy metadata

Một row là một coded source row. Trước khi giải thích chính sách, hãy đọc metadata: source title, issuing body, date, URL và access date.

In [2]:
policy = pd.read_csv(DATA_PATH)
print("Rows:", len(policy))
print("Documents:", policy["doc_id"].nunique())
print("Source types:", ", ".join(policy["source_type"].drop_duplicates()))
print(policy[["coding_id", "title", "issuing_body", "issue_date", "source_type", "policy_area"]].head(6).to_string(index=False))

Rows: 12
Documents: 6
Source types: policy_plan, statistical_bulletin, metadata_standard, data_portal, coding_model, policy_dashboard, comparative_report
coding_id                                                    title                            issuing_body issue_date          source_type          policy_area
     C001                      Education Powerhouse Plan 2024-2035 CPC Central Committee and State Council 2025-01-19          policy_plan    system_governance
     C002                      Education Powerhouse Plan 2024-2035 CPC Central Committee and State Council 2025-01-19          policy_plan       digitalization
     C003                      Education Powerhouse Plan 2024-2035 CPC Central Committee and State Council 2025-01-19          policy_plan  teacher_development
     C004                      Education Powerhouse Plan 2024-2035 CPC Central Committee and State Council 2025-01-19          policy_plan internationalization
     C005 2024 National Education Development 

## 2. Convert dates and build a timeline

`pd.to_datetime()` biến date text thành dữ liệu thời gian. Sau đó `sort_values()` giúp xếp nguồn theo thứ tự xuất hiện. Cột `date_basis` nhắc ta phân biệt publication date với access-date placeholder.

In [3]:
policy["issue_date_parsed"] = pd.to_datetime(policy["issue_date"], errors="coerce")
bad_dates = policy[policy["issue_date_parsed"].isna()][["coding_id", "issue_date"]]
if not bad_dates.empty:
    raise ValueError("Invalid issue_date values:\n" + bad_dates.to_string(index=False))
policy["issue_year"] = policy["issue_date_parsed"].dt.year
policy["source_label"] = policy["doc_id"] + " " + policy["source_type"]
timeline_columns = [
    "coding_id", "doc_id", "title", "issuing_body", "issue_date", "date_basis",
    "issue_date_parsed", "source_type", "policy_area", "theme_code", "url", "access_date",
]
timeline = policy[timeline_columns].sort_values(["issue_date_parsed", "coding_id"])
timeline.to_csv(TABLE_DIR / "week13_timeline.csv", index=False)
print(timeline[["coding_id", "doc_id", "issue_date", "date_basis", "source_type", "policy_area", "theme_code"]].to_string(index=False))

coding_id doc_id issue_date              date_basis          source_type          policy_area              theme_code
     C001   D001 2025-01-19        publication_date          policy_plan    system_governance          strategic_goal
     C002   D001 2025-01-19        publication_date          policy_plan       digitalization        digital_strategy
     C003   D001 2025-01-19        publication_date          policy_plan  teacher_development         teacher_quality
     C004   D001 2025-01-19        publication_date          policy_plan internationalization international_influence
     C005   D002 2025-06-11        publication_date statistical_bulletin education_statistics    statistical_snapshot
     C006   D002 2025-06-11        publication_date statistical_bulletin  teacher_development          teacher_supply
     C007   D002 2025-06-11        publication_date statistical_bulletin        equity_access   access_and_enrollment
     C012   D006 2025-11-28        publication_date   co

## 3. Count policy areas and source types

Đếm code không phải để chứng minh policy impact. Đếm để biết source set của mình đang thiên về chủ đề nào.

In [4]:
policy_area_summary = policy["policy_area"].value_counts().rename_axis("policy_area").reset_index(name="row_count")
policy_area_summary["percent"] = (policy_area_summary["row_count"] / len(policy) * 100).round(1)
policy_area_summary.to_csv(TABLE_DIR / "week13_policy_area_summary.csv", index=False)
print(policy_area_summary.to_string(index=False))
source_type_summary = policy["source_type"].value_counts().rename_axis("source_type").reset_index(name="row_count")
source_type_summary.to_csv(TABLE_DIR / "week13_source_type_summary.csv", index=False)
print()
print(source_type_summary.to_string(index=False))

         policy_area  row_count  percent
 teacher_development          2     16.7
  indicator_metadata          2     16.7
policy_coding_method          2     16.7
   system_governance          1      8.3
      digitalization          1      8.3
internationalization          1      8.3
education_statistics          1      8.3
       equity_access          1      8.3
  comparative_policy          1      8.3

         source_type  row_count
         policy_plan          4
statistical_bulletin          3
   metadata_standard          1
         data_portal          1
        coding_model          1
    policy_dashboard          1
  comparative_report          1


## 4. Stretch: cross-tab evidence type by policy area

`pd.crosstab()` giúp kiểm tra xem một policy area đến từ policy text, statistics, metadata hay coding model. Đây là bước chống overclaim, nhưng tuần này giữ ở mức Stretch để core workload nhẹ hơn.

In [5]:
theme_by_source = pd.crosstab(policy["policy_area"], policy["evidence_type"])
theme_by_source.to_csv(TABLE_DIR / "week13_theme_by_source_crosstab.csv")
print(theme_by_source.to_string())
sample = policy[["coding_id", "title", "policy_area", "theme_code", "evidence_type", "excerpt_paraphrase", "coder_note"]].head(8)
sample.to_csv(TABLE_DIR / "week13_policy_coding_sample.csv", index=False)
print()
print("Coding sample:")
print(sample.to_string(index=False))

evidence_type         metadata  method_model  milestone  policy_text  report  statistical_indicator
policy_area                                                                                        
comparative_policy           0             0          0            0       1                      0
digitalization               0             0          0            1       0                      0
education_statistics         0             0          0            0       0                      1
equity_access                0             0          0            0       0                      1
indicator_metadata           2             0          0            0       0                      0
internationalization         0             0          0            1       0                      0
policy_coding_method         0             2          0            0       0                      0
system_governance            0             0          1            0       0                      0


## 5. Export figures

Figure 1 là timeline source dates. Figure 2 là policy-area counts. Cả hai mô tả source set, không đánh giá hiệu quả chính sách.

In [6]:
plt.rcParams.update({"font.size": 11, "axes.titlesize": 14, "axes.labelsize": 11})
plot_timeline = timeline.drop_duplicates("doc_id").sort_values("issue_date_parsed")
fig, ax = plt.subplots(figsize=(9, 4.8))
colors = plot_timeline["date_basis"].map({"publication_date": "#2f63ea", "access_date_placeholder": "#bf5b0d"})
ax.scatter(plot_timeline["issue_date_parsed"], range(len(plot_timeline)), color=colors, s=90)
for y, (_, row) in enumerate(plot_timeline.iterrows()):
    marker = "pub" if row["date_basis"] == "publication_date" else "access"
    ax.text(row["issue_date_parsed"], y + 0.12, f"{row['doc_id']} · {marker}", fontsize=9, ha="center")
ax.set_yticks(range(len(plot_timeline)))
ax.set_yticklabels(plot_timeline["doc_id"] + " / " + plot_timeline["source_type"])
ax.set_title("Week 13 policy source timeline")
ax.set_xlabel("Publication date or access-date placeholder")
ax.set_ylabel("Source")
ax.grid(axis="x", color="#d8e2f0")
fig.tight_layout()
fig.savefig(FIG_DIR / "week13_policy_timeline.png", dpi=200)
fig.savefig(FIG_DIR / "week13_policy_timeline.svg", metadata={"Date": "2026-06-04"})
plt.close(fig)
plot_area = policy_area_summary.sort_values("row_count")
fig, ax = plt.subplots(figsize=(8.2, 5.2))
ax.barh(plot_area["policy_area"], plot_area["row_count"], color="#1f7a4d")
ax.set_title("Week 13 policy area counts")
ax.set_xlabel("Coded rows")
ax.set_ylabel("Policy area")
for i, value in enumerate(plot_area["row_count"]):
    ax.text(value + 0.05, i, str(value), va="center", fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "week13_policy_area_counts.png", dpi=200)
fig.savefig(FIG_DIR / "week13_policy_area_counts.svg", metadata={"Date": "2026-06-04"})
plt.close(fig)
print("Figures exported:")
print(FIG_DIR / "week13_policy_timeline.png")
print(FIG_DIR / "week13_policy_area_counts.png")

Figures exported:
weeks/week-13-education-policy-data/outputs/figures/week13_policy_timeline.png
weeks/week-13-education-policy-data/outputs/figures/week13_policy_area_counts.png


## 6. Paper-facing writing

Đoạn Data/Methods phải tách bốn ý: unit of analysis là gì, metadata/coding scheme gồm gì, date basis được xử lý thế nào, và limitation là gì.

In [7]:
source_types = ", ".join(source_type_summary["source_type"].tolist())
policy_areas = ", ".join(policy_area_summary["policy_area"].head(5).tolist())
access_placeholder_count = int((policy["date_basis"] == "access_date_placeholder").sum())
paragraph = (
    f"The Week 13 classroom dataset contains {len(policy)} synthetic, paraphrased coded rows drawn from {policy['doc_id'].nunique()} policy or source documents. "
    "The unit of analysis is one coded source entry, not one learner, school, or full policy corpus. "
    "Each row records source metadata, including title, issuing body, source type, URL, access date, and a coder note, plus analytic fields such as policy_area, theme_code, and evidence_type. "
    f"The source types include {source_types}, while the main coded policy areas include {policy_areas}. "
    f"Dates are parsed with pd.to_datetime; {access_placeholder_count} rows use access-date placeholders because the source pages are dynamic metadata, dashboard, or profile pages. "
    "Therefore, the timeline should be read as a source map for transparent paper writing, not as evidence of implementation sequence, learner outcomes, or causal policy effects."
)
word_count = len(paragraph.split())
print(paragraph)
print()
print("Word count:", word_count)
caption_timeline = "Figure 1 shows the timeline of 12 coded policy-source rows in the Week 13 classroom dataset. Color/labels distinguish publication dates from access-date placeholders; the figure describes source coverage rather than policy impact."
caption_bar = "Figure 2 shows policy-area counts in the Week 13 classroom dataset. Counts describe the small synthetic coding sample and should not be interpreted as policy importance or implementation quality."
source_note = (
    "Source note: policy/source metadata were checked on 2026-06-04. Core sources include the PRC State Council education blueprint page, the PRC MOE 2024 statistical bulletin, UNESCO UIS SDG 4 indicators, and UNESCO GEM PEER. "
    f"The classroom CSV is synthetic/paraphrased, SHA-256 {EXPECTED_SHA}, and should not be treated as a full policy corpus."
)
submission_text = f"""# Week 13 Submission Text Model

## Timeline Caption

{caption_timeline}

## Bar-Figure Caption

{caption_bar}

## Source Note

{source_note}

## Data/Methods Paragraph

{paragraph}
"""
(TABLE_DIR / "week13_submission_text.md").write_text(submission_text, encoding="utf-8")
print("Submission text exported:", TABLE_DIR / "week13_submission_text.md")
print()
print("Timeline caption:")
print(caption_timeline)
print()
print("Bar-figure caption:")
print(caption_bar)
print()
print("Source note:")
print(source_note)

The Week 13 classroom dataset contains 12 synthetic, paraphrased coded rows drawn from 6 policy or source documents. The unit of analysis is one coded source entry, not one learner, school, or full policy corpus. Each row records source metadata, including title, issuing body, source type, URL, access date, and a coder note, plus analytic fields such as policy_area, theme_code, and evidence_type. The source types include policy_plan, statistical_bulletin, metadata_standard, data_portal, coding_model, policy_dashboard, comparative_report, while the main coded policy areas include teacher_development, indicator_metadata, policy_coding_method, system_governance, digitalization. Dates are parsed with pd.to_datetime; 4 rows use access-date placeholders because the source pages are dynamic metadata, dashboard, or profile pages. Therefore, the timeline should be read as a source map for transparent paper writing, not as evidence of implementation sequence, learner outcomes, or causal policy e